In [1]:
!pip install langgraph typing_extensions IPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 105.0 MB/s eta 0:00:00


In [4]:
!pip install -qU langchain-nvidia-ai-endpoints


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 1.9 MB/s eta 0:00:00


In [14]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os

llm = ChatNVIDIA(model="openai/gpt-oss-20b",api_key=os.getenv("LLM_API"))

/usr/local/lib/python3.12/dist-packages/langchain_nvidia_ai_endpoints/_common.py:212: UserWarning: An API key is required for the hosted NIM. This will become an error in the future.
  warnings.warn(


In [3]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display


class State(TypedDict):
   topic:str
   explain:str
   example:str
   quiz:str






In [8]:
def explain_topic(state: State):
    """Explain topic in simple way"""

    msg = llm.invoke(
        f"Explain {state['topic']} in a simple and beginner-friendly way"
    )
    return {"explain": msg.content}

In [9]:
def example_topic(state: State):
    """Give real-world example"""

    msg = llm.invoke(
        f"Give a real-world example of {state['topic']} and explain how it is used in practice"
    )
    return {"example": msg.content}

In [10]:
def quiz_topic(state: State):
    """Generate quiz questions"""

    msg = llm.invoke(
        f"Generate 3 quiz questions with answers about {state['topic']}. Keep it simple and educational."
    )
    return {"quiz": msg.content}

In [12]:

workflow = StateGraph(State)

#  Nodes
workflow.add_node("explain_topic", explain_topic)
workflow.add_node("example_topic", example_topic)
workflow.add_node("quiz_topic", quiz_topic)

#  Flow (sequential chaining)
workflow.add_edge(START, "explain_topic")
workflow.add_edge("explain_topic", "example_topic")
workflow.add_edge("example_topic", "quiz_topic")
workflow.add_edge("quiz_topic", END)

#  Compile graph
app = workflow.compile()

In [13]:
result = app.invoke({"topic": "JWT authentication"})
print(result["explain"])
print(result["example"])
print(result["quiz"])

# JWT Authentication – A Beginner’s Story

Think of JWT (JSON Web Token) as a **paper‑style “pass”** that lets a computer (your browser, phone, or app) prove who it is to another computer (your back‑end server) without having to ask the back‑end for a new pass every time you make a request.

---

## 1. What is a JWT?

A JWT is just a string made of three parts, glued together with dots (`.`):

```
header.payload.signature
```

1. **Header** – tells the server *how* the token was signed (e.g. HS256 = HMAC‑SHA256).  
2. **Payload** – “claims” – data about the user you are sending (e.g. `userId: 123`, `exp: 1697990400`).  
3. **Signature** – a cryptographic checksum made with a secret key that guarantees the token hasn’t been tampered with.

The whole string looks like:

```
eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.
eyJ1c2VySWQiOjEyMywiZXhwIjoxNjk3OTkwNDAwfQ.
LmD3GxoV4lX67QeFR2DIc3K...

```

---

## 2. High‑level Flow

| Step | Who does it | What happens? |
|------|------------|--------------